In [2]:
import pandas as pd
import pm4py

single_log = pd.read_csv(
    "../../data/processed/CTB/s5_sample_20.000_eventlog_single_block_target_features.csv"
)

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    single_log[col] = pd.to_datetime(
        single_log[col]
    )



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




In [4]:
# =====================================================
# KEEP ONLY RELEVANT ATTRIBUTES
# =====================================================

columns_to_keep = [

    # mandatory PM columns
    "case:concept:name",
    "concept:name",
    "org:resource",

    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp",

    # strongest candidates from RF analysis
    "target_utilization",
    "target_demand",
    "visit_complexity",

    "n_containers",
    "n_receives",
    "n_deliveries"

]

single_log = single_log[
    columns_to_keep
].copy()

print(single_log.columns.tolist())
print()
print("Columns:", len(single_log.columns))

['case:concept:name', 'concept:name', 'org:resource', 'enabled:timestamp', 'start:timestamp', 'time:timestamp', 'target_utilization', 'target_demand', 'visit_complexity', 'n_containers', 'n_receives', 'n_deliveries']

Columns: 12


In [5]:
event_log_df = pm4py.format_dataframe(
    single_log,
    case_id="case:concept:name",
    activity_key="concept:name",
    timestamp_key="time:timestamp"
)

In [6]:
net, im, fm = pm4py.discover_petri_net_inductive(
    event_log_df
)

In [7]:
pm4py.write_pnml(
    net,
    im,
    fm,
    "reduced_baseline.pnml"
)

In [8]:
# Petri Net Visuals ohne Graphviz nicht möglich 

pm4py.save_vis_petri_net(
    net,
    im,
    fm,
    "reduced_baseline.png"
)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [9]:
from pm4py.objects.conversion.log import converter

event_log = converter.apply(
    event_log_df,
    variant=converter.Variants.TO_EVENT_LOG
)

In [10]:
from prosit.simulator import (
    SimulatorParameters
)

params = SimulatorParameters(
    net,
    im,
    fm
)

params.discover_from_eventlog(
    event_log,
    max_depth_tree=10
)

Resources discovery...
Data attributes discovery...
PATCHED 1.0 script is running
Feature discovery...


c:\Users\Grigat-J\AppData\Local\miniconda3\envs\prosit_test\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 20000/20000 [00:23<00:00, 842.93it/s]


Transition Probabilities discovery...


transition models: 100%|██████████| 43/43 [00:31<00:00,  1.35it/s]


Resource Weights discovery...


...

Calendars discovery...
Execution Time discovery...


exec-time models: 100%|██████████| 11/11 [00:08<00:00,  1.33it/s]


Waiting Time discovery...


waiting-time models: 100%|██████████| 26/26 [00:11<00:00,  2.19it/s]


Arrival Time discovery...


In [11]:
model = params.execution_time_distributions[

"RMG_receive"

]

print(type(model.decision_tree))

AttributeError: 'DecisionRules' object has no attribute 'decision_tree'

In [ ]:
print(params.resources)

print()

print(
    params.attribute_values_label_categorical
)

['Res.GateIn', 'Res.GateOut', 'LL', 'T13', 'T14', 'T24', 'T16', 'T17', 'T21', 'T12', 'T22', 'T20', 'T23', 'T11', 'T27', 'T10', 'T06', 'T18', 'T09', 'T15', 'T19', 'T07', 'T08', 'T26', 'T25', 'HO2']

{}


In [ ]:
import pickle

with open(
    "reduced_baseline_depth10.pkl",
    "wb"
) as f:

    pickle.dump(
        params,
        f
    )

params.to_json(
    "reduced_baseline_depth10.json"
)

In [ ]:
print(
    params.execution_time_distributions[
        "RMG_receive"
    ].rules
)


{0: {'value': 12.13358880670172, 'dist': (<scipy.stats._continuous_distns.lognorm_gen object at 0x000001B252A73C40>, (0.712679107241122, -0.11726334808684913, 9.287638679555458), 0.0, 215.0), 'sampled': [19.09586097256402, 6.509481228767251, 18.891500773465694, 28.541407742129145, 7.2078374156235325, 2.45872634938638, 15.588945447718507, 9.008548514120047, 23.194524524104335, 5.117120919288571, 16.743817018511347, 4.112028521305619, 15.002420376950845, 48.532713654478194, 3.36656289714829, 8.574632258663577, 5.818309573870541, 7.981100041837815, 17.347864887808658, 25.27891031167025, 6.680688203265587, 14.71793545700246, 23.65292084080262, 18.001860960594815, 12.9184268240729, 6.045169080035725, 6.995343912365903, 5.209477319930603, 11.077014016441613, 7.172239673563759, 9.896528477810184, 32.17008299015185, 21.279747684590394, 13.76642757951126, 4.801183489005964, 13.681934399613436, 35.43059787189072, 17.975395690905838, 10.534259610322389, 16.19510364496103, 4.909121473978626, 17.46

In [ ]:
params.execution_time_distributions["RMG_mixed"]

In [ ]:
print(

params.execution_time_distributions.keys()

)
for act, model in (
    params.execution_time_distributions.items()
):

    print(
        act,
        len(model.rules)
    )


dict_keys(['LL_receive', 'HO2_mixed', 'HO2_receive', 'LL_mixed', 'LL_delivery', 'HO2_delivery', 'Gate In', 'Gate Out', 'RMG_mixed', 'RMG_delivery', 'RMG_receive'])
LL_receive 1
HO2_mixed 1
HO2_receive 1
LL_mixed 1
LL_delivery 1
HO2_delivery 1
Gate In 1
Gate Out 1
RMG_mixed 1
RMG_delivery 1
RMG_receive 1


In [15]:
print(
    params.distribution_data_attributes.keys()
)
print(
    params.distribution_data_attributes
)

dict_keys(['mode', 'data'])
{'mode': 'distribution', 'data': {'target_demand': {'type': 'continuous', 'dist': 'norm', 'params': [6.0180925, 2.995811537203859], 'min': 0.0, 'max': 19.9, 'mean': 6.0180925}, 'visit_complexity': {'type': 'continuous', 'dist': 'lognorm', 'params': [0.6592106578855956, 1.5458512357981378, 3.8715562365509477], 'min': 2.0, 'max': 26.0, 'mean': 6.38145}, 'n_containers': {'type': 'continuous', 'dist': 'gamma', 'params': [0.17360907071359538, 0.9999999999999999, 0.323725569836006], 'min': 1.0, 'max': 4.0, 'mean': 1.18385}, '@@case_index': {'type': 'continuous', 'dist': 'uniform', 'params': [0.0, 19999.0], 'min': 0.0, 'max': 19999.0, 'mean': 9999.5}, '@@index': {'type': 'continuous', 'dist': 'uniform', 'params': [0.0, 61715.0], 'min': 0.0, 'max': 61715.0, 'mean': 30777.93015}, 'target_utilization': {'type': 'continuous', 'dist': 'norm', 'params': [0.613533545, 0.13135620878067766], 'min': 0.0071, 'max': 0.8385, 'mean': 0.613533545}, 'n_deliveries': {'type': 'conti

In [16]:
model = params.execution_time_distributions[
    "RMG_receive"
]

print(type(model))


<class 'prosit.utils.rule_utils.DecisionRules'>


In [17]:

print(model.graph)


None


In [ ]:
len(
params.execution_time_distributions[
"RMG_receive"
].rules)


1

In [19]:
len(
params.execution_time_distributions[
"RMG_delivery"
].rules)

1

In [49]:
model.write_dot("rmg_receive.dot")

In [20]:
model = params.execution_time_distributions[
    "RMG_receive"
]

print(model.rules)

{0: {'value': 12.13358880670172, 'dist': (<scipy.stats._continuous_distns.lognorm_gen object at 0x000001B252A73C40>, (0.712679107241122, -0.11726334808684913, 9.287638679555458), 0.0, 215.0), 'sampled': [19.09586097256402, 6.509481228767251, 18.891500773465694, 28.541407742129145, 7.2078374156235325, 2.45872634938638, 15.588945447718507, 9.008548514120047, 23.194524524104335, 5.117120919288571, 16.743817018511347, 4.112028521305619, 15.002420376950845, 48.532713654478194, 3.36656289714829, 8.574632258663577, 5.818309573870541, 7.981100041837815, 17.347864887808658, 25.27891031167025, 6.680688203265587, 14.71793545700246, 23.65292084080262, 18.001860960594815, 12.9184268240729, 6.045169080035725, 6.995343912365903, 5.209477319930603, 11.077014016441613, 7.172239673563759, 9.896528477810184, 32.17008299015185, 21.279747684590394, 13.76642757951126, 4.801183489005964, 13.681934399613436, 35.43059787189072, 17.975395690905838, 10.534259610322389, 16.19510364496103, 4.909121473978626, 17.46

In [21]:
print(
    model.rules[0].keys()
)

dict_keys(['value', 'dist', 'sampled'])


In [25]:
model = params.execution_time_distributions["RMG_receive"]
print(type(model))

print(vars(model))
print(model.__dict__.keys())

<class 'prosit.utils.rule_utils.DecisionRules'>
{'rules': {0: {'value': 12.13358880670172, 'dist': (<scipy.stats._continuous_distns.lognorm_gen object at 0x000001B252A73C40>, (0.712679107241122, -0.11726334808684913, 9.287638679555458), 0.0, 215.0), 'sampled': [19.09586097256402, 6.509481228767251, 18.891500773465694, 28.541407742129145, 7.2078374156235325, 2.45872634938638, 15.588945447718507, 9.008548514120047, 23.194524524104335, 5.117120919288571, 16.743817018511347, 4.112028521305619, 15.002420376950845, 48.532713654478194, 3.36656289714829, 8.574632258663577, 5.818309573870541, 7.981100041837815, 17.347864887808658, 25.27891031167025, 6.680688203265587, 14.71793545700246, 23.65292084080262, 18.001860960594815, 12.9184268240729, 6.045169080035725, 6.995343912365903, 5.209477319930603, 11.077014016441613, 7.172239673563759, 9.896528477810184, 32.17008299015185, 21.279747684590394, 13.76642757951126, 4.801183489005964, 13.681934399613436, 35.43059787189072, 17.975395690905838, 10.53

In [26]:
print(model.apply)
print(model.from_decision_tree)
print(model.from_river_decision_tree)

<bound method DecisionRules.apply of <prosit.utils.rule_utils.DecisionRules object at 0x000001B25D345540>>
<bound method DecisionRules.from_decision_tree of <prosit.utils.rule_utils.DecisionRules object at 0x000001B25D345540>>
<bound method DecisionRules.from_river_decision_tree of <prosit.utils.rule_utils.DecisionRules object at 0x000001B25D345540>>


In [27]:
import inspect

print(
    inspect.getsource(
        model.__class__
    )
)

class DecisionRules:
    def __init__(self):
        self.rules = None
        self.graph = None

    def from_decision_tree(self, decision_tree):
        self.decision_tree = decision_tree
        self.graph = build_graph_vis(decision_tree, True)
        nodes, edges = parse_tree(self.graph.source)
        self.rules = build_tree_structure(nodes, edges)

    def from_river_decision_tree(self, decision_tree, distribution=False, min_value=0, max_value=MAX_DURATION_MINUTES):
        self.decision_tree = decision_tree
        self.rules = transform_river_decision_tree_data(decision_tree, distribution, min_value, max_value)

    def apply(self, features):
        return traverse_tree(self.rules, features)
    
    def apply_distribution(self, features):
        return traverse_tree_distribution(self.rules, features)
    
    def write_dot(self, file_name='decision_tree.dot'):
        if self.graph:
            self.graph.render(file_name)



In [28]:
import inspect

print(
    inspect.getfile(
        model.__class__
    )
)


c:\Users\Grigat-J\AppData\Local\miniconda3\envs\prosit_test\lib\site-packages\prosit\utils\rule_utils.py
